# 面试题：如何从零实现 MobileNet 与 DenseNet，并说明它们为何高效？

## 面试回答主线

MobileNet 把标准卷积分解为“逐通道 depthwise 空间卷积 + 1×1 pointwise 通道混合”，主要计算量从 $K^2C_{in}C_{out}$ 降到 $K^2C_{in}+C_{in}C_{out}$。DenseNet 让每层接收此前全部特征并把新特征沿通道拼接，改善梯度与特征复用，但激活内存会随层数增长。面试时除了写 `class` 与 `forward`，还应能推导参数量、展示中间 shape、解释 groups 和 concat 约束。下面只用 `nn.Parameter` 与底层 `F.conv2d` 手写两个微型网络，并在真实可读的条纹质检任务上训练。

## 真实案例：流水线图像方向质检

相机截取 16×16 RGB 小图，目标判断亮色缺陷是竖向划痕还是横向裂纹。我们离线生成不同位置、宽度、颜色增益和传感器噪声的 28 张结构化样本，其中 20 张训练、8 张测试；它不冒充真实工业数据，但保留了空间方向、平移和噪声三种决策因素。

In [1]:
import math  # 导入平方根用于卷积参数初始化。
import torch  # 导入 PyTorch 以生成图像并真实训练手写网络。
from torch import nn  # 导入基础模块和参数容器。
import torch.nn.functional as F  # 导入底层 conv2d、交叉熵与 padding。
torch.set_num_threads(1)  # 固定小图实验为单线程以快速稳定复现。
torch.manual_seed(41)  # 固定图像噪声与模型初始化。
def make_stripe(label, position, width, noise_seed):  # 生成带位置、宽度与噪声变化的三通道缺陷图。
    generator = torch.Generator().manual_seed(noise_seed)  # 为每张图建立确定性噪声源。
    image = torch.randn(3, 16, 16, generator=generator) * 0.06  # 创建低幅传感器背景噪声。
    if label == 0:  # 类别零表示竖向划痕。
        image[:, :, position:position + width] += torch.tensor([1.0, 0.8, 0.6])[:, None, None]  # 在指定列写入带颜色增益的竖条。
    else:  # 类别一表示横向裂纹。
        image[:, position:position + width, :] += torch.tensor([1.0, 0.8, 0.6])[:, None, None]  # 在指定行写入相同总亮度横条。
    return image.clamp(-0.2, 1.2)  # 限制极端噪声同时保留可学习对比度。
samples = []  # 收集图像、标签与可读元数据。
for label in (0, 1):  # 分别生成竖向和横向两类。
    for local_index in range(14):  # 每类生成十四个位置与噪声组合。
        position = 2 + (local_index * 3) % 10  # 让条纹覆盖多个空间位置。
        width = 1 + local_index % 2  # 在一像素与两像素宽度之间切换。
        image = make_stripe(label, position, width, 100 + label * 50 + local_index)  # 生成当前确定性样本。
        samples.append((image, label, position, width))  # 保存张量和业务可读字段。
train_indices = list(range(10)) + list(range(14, 24))  # 每类取十张作为训练集。
test_indices = list(range(10, 14)) + list(range(24, 28))  # 每类留四张位置组合作为测试集。
train_images = torch.stack([samples[index][0] for index in train_indices])  # 堆叠二十张训练图像。
train_targets = torch.tensor([samples[index][1] for index in train_indices])  # 创建训练标签张量。
test_images = torch.stack([samples[index][0] for index in test_indices])  # 堆叠八张独立测试图像。
test_targets = torch.tensor([samples[index][1] for index in test_indices])  # 创建测试标签张量。
label_names = ["竖向划痕", "横向裂纹"]  # 定义两类质检语义名称。
print("集合  编号  标签      位置  宽度  全图均值  最大值")  # 输出输入预览表头。
for split, indices in (("训练", train_indices[:3]), ("测试", test_indices)):  # 展示三条训练样本和全部测试样本。
    for index in indices:  # 遍历选中的可读案例。
        image, label, position, width = samples[index]  # 解包图像及生成元数据。
        print(f"{split:<4} {index:>3}   {label_names[label]:<6} {position:>3}   {width:>2}   {float(image.mean()):.4f}   {float(image.max()):.3f}")  # 展示空间变化与近似相同亮度。

集合  编号  标签      位置  宽度  全图均值  最大值
训练     0   竖向划痕     2    1   0.0518   1.074
训练     1   竖向划痕     5    2   0.1005   1.143
训练     2   竖向划痕     8    1   0.0508   1.130
测试    10   竖向划痕     2    1   0.0548   1.164
测试    11   竖向划痕     5    2   0.1014   1.154
测试    12   竖向划痕     8    1   0.0483   1.066
测试    13   竖向划痕    11    2   0.0979   1.149
测试    24   横向裂纹     2    1   0.0497   1.074
测试    25   横向裂纹     5    2   0.0982   1.088
测试    26   横向裂纹     8    1   0.0486   1.145
测试    27   横向裂纹    11    2   0.1012   1.110


## Baseline（基线）：只看全图平均亮度

竖条和横条包含几乎相同数量的亮像素，所以空间平均会抹掉方向。这里用训练集平均亮度的中位数做阈值，亮于阈值预测横纹；它有真实输入和指标，但缺少空间归纳偏置。

In [2]:
brightness_threshold = float(train_images.mean(dim=(1, 2, 3)).median())  # 从训练集确定单一亮度阈值。
baseline_scores = test_images.mean(dim=(1, 2, 3))  # 把每张测试图压缩为全图平均亮度。
baseline_predictions = (baseline_scores > brightness_threshold).long()  # 根据亮度阈值输出横纹或竖纹。
baseline_accuracy = float((baseline_predictions == test_targets).float().mean())  # 计算八张测试图准确率。
print(f"训练亮度中位阈值：{brightness_threshold:.4f}")  # 展示基线决策边界。
print("样本  gold      平均亮度  基线预测  正确")  # 输出逐样本基线结果表头。
for row, index in enumerate(test_indices):  # 遍历全部留出测试样本。
    gold_name = label_names[int(test_targets[row])]  # 还原标准标签名称。
    predicted_name = label_names[int(baseline_predictions[row])]  # 还原基线预测名称。
    print(f"{index:>3}   {gold_name:<6} {float(baseline_scores[row]):.4f}    {predicted_name:<6} {bool(baseline_predictions[row] == test_targets[row])}")  # 展示亮度重叠造成的错误。
print(f"亮度基线测试准确率：{baseline_accuracy:.1%}")  # 输出神经网络要比较的同口径测试指标。

训练亮度中位阈值：0.0536
样本  gold      平均亮度  基线预测  正确
 10   竖向划痕   0.0548    横向裂纹   False
 11   竖向划痕   0.1014    横向裂纹   False
 12   竖向划痕   0.0483    竖向划痕   True
 13   竖向划痕   0.0979    横向裂纹   False
 24   横向裂纹   0.0497    竖向划痕   False
 25   横向裂纹   0.0982    横向裂纹   True
 26   横向裂纹   0.0486    竖向划痕   False
 27   横向裂纹   0.1012    横向裂纹   True
亮度基线测试准确率：37.5%


## 核心实现一：手写 depthwise-separable MobileNet

depthwise 权重形状是 `[输入通道, 1, K, K]`，`groups=输入通道` 保证每个通道独立做空间卷积；pointwise 权重形状是 `[输出通道, 输入通道, 1, 1]`，负责跨通道组合。这里不用 `nn.Conv2d`、`nn.MobileNet` 或现成 block。

In [3]:
class DepthwiseSeparableBlock(nn.Module):  # 定义深度卷积加逐点卷积的高效块。
    def __init__(self, input_channels, output_channels, stride=1):  # 初始化空间核、通道混合核与偏置。
        super().__init__()  # 注册基础模块状态。
        depth_scale = math.sqrt(2.0 / 9.0)  # 计算三乘三深度核的 He 缩放。
        point_scale = math.sqrt(2.0 / input_channels)  # 计算一点卷积的 He 缩放。
        self.depth_weight = nn.Parameter(torch.randn(input_channels, 1, 3, 3) * depth_scale)  # 为每个输入通道创建独立三乘三核。
        self.depth_bias = nn.Parameter(torch.zeros(input_channels))  # 创建逐输入通道偏置。
        self.point_weight = nn.Parameter(torch.randn(output_channels, input_channels, 1, 1) * point_scale)  # 创建跨通道一点卷积核。
        self.point_bias = nn.Parameter(torch.zeros(output_channels))  # 创建输出通道偏置。
        self.stride = stride  # 保存空间下采样步长。
    def forward(self, inputs):  # 执行 depthwise、ReLU、pointwise 与 ReLU。
        depth = F.conv2d(inputs, self.depth_weight, self.depth_bias, stride=self.stride, padding=1, groups=inputs.shape[1])  # 对各通道分别提取空间纹理。
        depth = torch.relu(depth)  # 对空间响应应用非线性。
        mixed = F.conv2d(depth, self.point_weight, self.point_bias)  # 用一点卷积学习通道组合。
        return torch.relu(mixed)  # 返回高效卷积块输出。
class TinyMobileNet(nn.Module):  # 定义由两个手写高效块组成的微型分类网络。
    def __init__(self, class_count=2):  # 初始化两级特征与线性分类头。
        super().__init__()  # 注册基础模块状态。
        self.first = DepthwiseSeparableBlock(3, 8, stride=1)  # 把 RGB 特征扩展到八通道。
        self.second = DepthwiseSeparableBlock(8, 12, stride=2)  # 下采样并扩展到十二通道。
        self.output_weight = nn.Parameter(torch.randn(12, class_count) * 0.15)  # 创建空间汇聚后的类别权重。
        self.output_bias = nn.Parameter(torch.zeros(class_count))  # 创建类别偏置。
    def forward(self, images, return_features=False):  # 执行两级高效卷积与全局最大池化。
        first_features = self.first(images)  # 提取浅层方向边缘。
        second_features = self.second(first_features)  # 提取下采样后的组合纹理。
        pooled = second_features.amax(dim=(2, 3))  # 每通道保留最强空间响应以适配条纹平移。
        logits = pooled @ self.output_weight + self.output_bias  # 计算横纹与竖纹分数。
        if return_features:  # 教学观察模式返回中间特征。
            return logits, first_features, second_features, pooled  # 暴露每层 shape 与池化结果。
        return logits  # 普通训练模式只返回类别分数。
torch.manual_seed(43)  # 固定 MobileNet 参数初始化。
mobilenet = TinyMobileNet()  # 创建手写微型 MobileNet。
mobile_logits, mobile_first, mobile_second, mobile_pooled = mobilenet(train_images[:2], return_features=True)  # 对两张图运行完整前向过程。
standard_parameters = 3 * 3 * 3 * 8  # 计算三乘三标准卷积从三到八通道的权重数。
separable_parameters = 3 * 3 * 3 + 3 * 8  # 计算对应 depthwise 加 pointwise 权重数。
print("输入 -> block1 -> block2 -> 池化：", tuple(train_images[:2].shape), tuple(mobile_first.shape), tuple(mobile_second.shape), tuple(mobile_pooled.shape))  # 展示真实特征图形状变化。
print(f"首层标准卷积权重={standard_parameters}，可分离卷积权重={separable_parameters}，缩减={(1 - separable_parameters / standard_parameters):.1%}")  # 用公式量化参数节省。
print("首图池化特征前五维：", [round(float(value), 3) for value in mobile_pooled[0, :5]])  # 展示分类头实际读取的中间表示。

输入 -> block1 -> block2 -> 池化： (2, 3, 16, 16) (2, 8, 16, 16) (2, 12, 8, 8) (2, 12)
首层标准卷积权重=216，可分离卷积权重=51，缩减=76.4%
首图池化特征前五维： [1.305, 0.765, 0.032, 0.067, 0.003]


## 核心实现二：手写 DenseNet 的 dense connectivity

每个 DenseLayer 只生成 `growth_rate` 个新通道，DenseBlock 在 forward 中把输入和所有新特征 `cat` 在一起。因此第 $l$ 层的输入通道数是 $C_0+l	imes growth$。这不是残差相加：拼接会保留原始通道，并让后续层自己选择复用哪些层。

In [4]:
class DenseLayer(nn.Module):  # 定义生成固定 growth 新特征的手写稠密层。
    def __init__(self, input_channels, growth_rate):  # 初始化三乘三卷积权重与偏置。
        super().__init__()  # 注册基础模块状态。
        scale = math.sqrt(2.0 / (input_channels * 9))  # 根据输入扇入计算 He 缩放。
        self.weight = nn.Parameter(torch.randn(growth_rate, input_channels, 3, 3) * scale)  # 创建读取全部历史通道的卷积核。
        self.bias = nn.Parameter(torch.zeros(growth_rate))  # 创建 growth 个新通道偏置。
    def forward(self, features):  # 从累计特征生成新的 growth 通道。
        return torch.relu(F.conv2d(features, self.weight, self.bias, padding=1))  # 执行底层卷积并返回新特征。
class DenseBlock(nn.Module):  # 定义逐层拼接而不是覆盖的稠密块。
    def __init__(self, input_channels, growth_rate, layer_count):  # 按递增输入通道创建所有层。
        super().__init__()  # 注册基础模块状态。
        self.layers = nn.ModuleList([DenseLayer(input_channels + index * growth_rate, growth_rate) for index in range(layer_count)])  # 让第 l 层看到此前全部通道。
    def forward(self, inputs, return_widths=False):  # 逐层生成并拼接新特征。
        features = inputs  # 从原始图像或上一 block 特征开始。
        widths = [features.shape[1]]  # 记录每次拼接后的通道数。
        for layer in self.layers:  # 顺序执行所有稠密层。
            new_features = layer(features)  # 让当前层读取完整历史特征。
            features = torch.cat([features, new_features], dim=1)  # 沿通道保留旧特征并加入新特征。
            widths.append(features.shape[1])  # 保存本层后的累计宽度。
        if return_widths:  # 教学模式需要观察增长轨迹。
            return features, widths  # 返回累计特征与通道列表。
        return features  # 普通模式只返回累计特征。
class TinyDenseNet(nn.Module):  # 定义一个手写稠密块的微型分类网络。
    def __init__(self, class_count=2):  # 初始化两层 growth=5 的 block 与分类头。
        super().__init__()  # 注册基础模块状态。
        self.block = DenseBlock(3, 5, 3)  # 让通道数按三、八、十三、十八增长。
        self.output_weight = nn.Parameter(torch.randn(18, class_count) * 0.15)  # 创建累计特征到类别的权重。
        self.output_bias = nn.Parameter(torch.zeros(class_count))  # 创建类别偏置。
    def forward(self, images, return_features=False):  # 执行稠密连接、全局最大池化与分类。
        features, widths = self.block(images, return_widths=True)  # 获取全部历史特征和增长轨迹。
        pooled = features.amax(dim=(2, 3))  # 对每个旧/新通道做空间最大池化。
        logits = pooled @ self.output_weight + self.output_bias  # 计算二类质检分数。
        if return_features:  # 教学观察模式返回完整中间过程。
            return logits, features, pooled, widths  # 暴露拼接后的特征和通道轨迹。
        return logits  # 普通训练模式只返回类别分数。
torch.manual_seed(47)  # 固定 DenseNet 参数初始化。
densenet = TinyDenseNet()  # 创建手写微型 DenseNet。
dense_logits, dense_features, dense_pooled, dense_widths = densenet(train_images[:2], return_features=True)  # 对两张训练图执行一次前向传播。
dense_parameter_count = sum(parameter.numel() for parameter in densenet.parameters())  # 统计手写 DenseNet 的全部可学习参数。
mobile_parameter_count = sum(parameter.numel() for parameter in mobilenet.parameters())  # 统计手写 MobileNet 的全部可学习参数。
print("DenseBlock 通道增长轨迹：", dense_widths)  # 展示每层追加 growth 通道的真实结果。
print("DenseNet 累计特征形状：", tuple(dense_features.shape))  # 展示 concat 后最终特征张量。
print(f"微型模型参数量：MobileNet={mobile_parameter_count}，DenseNet={dense_parameter_count}")  # 对比两种结构的参数成本。

DenseBlock 通道增长轨迹： [3, 8, 13, 18]
DenseNet 累计特征形状： (2, 18, 16, 16)
微型模型参数量：MobileNet=276，DenseNet=1133


## 真实训练、留出测试与结果表

两个网络在同样 20 张训练图上用交叉熵优化，最后只在未参与更新的 8 张图上比较。输出 loss、首层梯度和逐图概率，避免用一个 assert 冒充实验。

In [5]:
def train_and_evaluate(model, epochs=180):  # 定义共享的训练与留出评估流程。
    optimizer = torch.optim.Adam(model.parameters(), lr=0.018)  # 创建直接更新手写卷积参数的 Adam。
    history = []  # 保存 loss、训练准确率和梯度范数。
    for epoch in range(epochs):  # 重复优化小型图像数据。
        optimizer.zero_grad()  # 清除上一轮累计梯度。
        logits = model(train_images)  # 调用手写 forward 得到训练分数。
        loss = F.cross_entropy(logits, train_targets)  # 计算两类交叉熵。
        loss.backward()  # 把误差反向传播到每个空间卷积核。
        first_parameter = next(model.parameters())  # 取得首层权重以观察梯度是否流入。
        gradient_norm = float(first_parameter.grad.norm().detach())  # 计算首层梯度二范数。
        optimizer.step()  # 按真实梯度更新网络参数。
        train_accuracy = float((logits.argmax(dim=-1) == train_targets).float().mean().detach())  # 计算当前轮训练准确率。
        history.append((float(loss.detach()), train_accuracy, gradient_norm))  # 保存训练轨迹。
    with torch.no_grad():  # 关闭留出集评估的梯度记录。
        probabilities = torch.softmax(model(test_images), dim=-1)  # 计算每张测试图的二类概率。
    predictions = probabilities.argmax(dim=-1)  # 取最大概率作为测试预测。
    test_accuracy = float((predictions == test_targets).float().mean())  # 计算留出测试准确率。
    return history, probabilities, predictions, test_accuracy  # 返回过程、逐图结果和指标。
mobile_history, mobile_probabilities, mobile_predictions, mobile_accuracy = train_and_evaluate(mobilenet)  # 训练并评估手写 MobileNet。
dense_history, dense_probabilities, dense_predictions, dense_accuracy = train_and_evaluate(densenet)  # 训练并评估手写 DenseNet。
print("模型          首轮loss  末轮loss  首层梯度  测试准确率")  # 输出同口径训练与测试表头。
print(f"亮度Baseline     -         -         -       {baseline_accuracy:.1%}")  # 展示无空间信息的基线。
print(f"MobileNet      {mobile_history[0][0]:.4f}    {mobile_history[-1][0]:.4f}    {mobile_history[0][2]:.4f}    {mobile_accuracy:.1%}")  # 展示高效卷积的优化与测试结果。
print(f"DenseNet       {dense_history[0][0]:.4f}    {dense_history[-1][0]:.4f}    {dense_history[0][2]:.4f}    {dense_accuracy:.1%}")  # 展示稠密连接的优化与测试结果。
print("样本 gold      MobileNet(概率)       DenseNet(概率)")  # 输出逐图预测表头。
for row, index in enumerate(test_indices):  # 遍历八张留出测试图。
    mobile_name = label_names[int(mobile_predictions[row])]  # 还原 MobileNet 预测标签。
    dense_name = label_names[int(dense_predictions[row])]  # 还原 DenseNet 预测标签。
    mobile_confidence = float(mobile_probabilities[row, mobile_predictions[row]])  # 读取 MobileNet 最大类别概率。
    dense_confidence = float(dense_probabilities[row, dense_predictions[row]])  # 读取 DenseNet 最大类别概率。
    print(f"{index:>3}   {label_names[int(test_targets[row])]:<6} {mobile_name}({mobile_confidence:.3f})       {dense_name}({dense_confidence:.3f})")  # 展示方向分类与置信度。

模型          首轮loss  末轮loss  首层梯度  测试准确率
亮度Baseline     -         -         -       37.5%
MobileNet      0.7024    0.0000    0.6483    100.0%
DenseNet       0.6471    0.0000    1.0019    100.0%
样本 gold      MobileNet(概率)       DenseNet(概率)
 10   竖向划痕   竖向划痕(1.000)       竖向划痕(1.000)
 11   竖向划痕   竖向划痕(1.000)       竖向划痕(1.000)
 12   竖向划痕   竖向划痕(1.000)       竖向划痕(1.000)
 13   竖向划痕   竖向划痕(1.000)       竖向划痕(1.000)
 24   横向裂纹   横向裂纹(1.000)       横向裂纹(1.000)
 25   横向裂纹   横向裂纹(1.000)       横向裂纹(1.000)
 26   横向裂纹   横向裂纹(1.000)       横向裂纹(1.000)
 27   横向裂纹   横向裂纹(1.000)       横向裂纹(1.000)


## 结果解读

亮度基线把空间维全部平均，因此面对总亮度相近的横纹和竖纹接近随机；两个手写网络则通过空间卷积学到方向。MobileNet 的首层参数推导直观看到 depthwise separable 的节省，DenseNet 的 `[3, 8, 13, 18]` 轨迹则证明每层都保留并读取历史通道。留出集结果仍只是受控教学实验：图片简单、训练分布与测试分布接近，不代表真实产线精度。

## 失败案例：depthwise 权重第二维写错

`groups=C` 时，PyTorch 要求每个输出通道只连接一个输入通道，因此权重必须是 `[C, 1, K, K]`。把它误写成标准卷积的 `[C, C, K, K]` 会直接触发通道契约错误。下面捕获错误消息，再用正确权重完成前向。

In [6]:
failure_input = torch.randn(1, 3, 8, 8)  # 构造三通道故障复现输入。
wrong_depth_weight = torch.randn(3, 3, 3, 3)  # 错把 depthwise 核写成每个输出连接三通道。
failure_message = ""  # 准备保存底层卷积抛出的可读错误。
try:  # 主动执行错误 groups 合同以复现生产常见配置故障。
    F.conv2d(failure_input, wrong_depth_weight, padding=1, groups=3)  # 使用 groups=3 但提供非法第二维。
except RuntimeError as error:  # 捕获预期的通道契约异常而不中断 Notebook。
    failure_message = str(error).split("\n")[0]  # 只保留首行以便教学阅读。
correct_depth_weight = torch.randn(3, 1, 3, 3)  # 修复为每个输出通道只含一个输入通道核。
fixed_output = F.conv2d(failure_input, correct_depth_weight, padding=1, groups=3)  # 使用正确权重完成真正 depthwise 前向。
wrong_parameter_count = wrong_depth_weight.numel()  # 统计错误权重的冗余参数量。
correct_parameter_count = correct_depth_weight.numel()  # 统计正确 depthwise 参数量。
print("错误复现：", failure_message)  # 展示真实运行时错误而不是只写概念结论。
print(f"修复后输出形状={tuple(fixed_output.shape)}，权重参数 {wrong_parameter_count} -> {correct_parameter_count}")  # 展示修复后的可执行结果与成本变化。

错误复现： Given groups=3, weight of size [3, 3, 3, 3], expected input[1, 3, 8, 8] to have 9 channels, but got 3 channels instead
修复后输出形状=(1, 3, 8, 8)，权重参数 81 -> 27


## 生产差距与追问

真实质检还要处理相机域偏移、尺度旋转、类别不平衡、标注噪声和缺陷定位；应报告按产线/日期分组的召回率、误报率、延迟、显存和 MACs，而不只看参数量。MobileNet 的瓶颈常来自内存访问与算子融合，DenseNet 的瓶颈常来自历史特征拼接造成的激活内存；量化、剪枝和部署后算子支持也必须实测。

## 最小回归测试

In [7]:
assert len(samples) >= 5  # 保证图像实验拥有足够多可读案例。
assert mobile_accuracy > baseline_accuracy  # 保护空间模型确实优于亮度基线。
assert dense_accuracy > baseline_accuracy  # 保护稠密连接确实利用空间结构。
assert dense_widths == [3, 8, 13, 18]  # 保护 DenseBlock 每层按 growth 追加通道。
assert tuple(fixed_output.shape) == (1, 3, 8, 8)  # 保护修复后的 depthwise 合同可执行。
print("最小回归测试通过：留出分类、通道增长与 depthwise groups 修复均保持有效。")  # 输出集中测试结论。

最小回归测试通过：留出分类、通道增长与 depthwise groups 修复均保持有效。
